## Machine Learning Group Project -  Transit Reliability and Neighbourhood Equity in Toronto

Hi Team, since I proposed this TTC delay in Toronto Neighbourhood project, I wanted to make sure I can access all the data, and that the data links up so we can build the dataset we need.

I am developing on Google Colab, so I have to make sure certain modules are installed on the machine I'm running on.

In [176]:
# Install the Toronto Open Data wrapper
!pip install toronto-open-data

# Install Geopandas (crucial for the spatial join)
!pip install geopandas

This next is specific to running on Google Colab.

In [177]:
# in google colab, mount the google drive to access the data
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# TTC Bus Delay Data



I downloaded the TTC Bus Delay Data off of the following website:

https://open.toronto.ca/dataset/ttc-bus-delay-data/

I downloaded the
"TTC Bus Delay Data since 2025" in JSON Format

In [178]:
import pandas as pd
import json

# This is my subway delay data
pathToGroup = r"/content/drive/My Drive/Colab Notebooks/GroupProject"
pathToFile = r"/content/drive/My Drive/Colab Notebooks/GroupProject/TorontoDataFiles/"

fileName = 'TTC Bus Delay Data since 2025.json'

delay_dataset = pd.read_json(pathToFile+fileName)

delay_dataset.head(10)

,_id,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle
0,1,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442
1,2,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,None,0
2,3,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,None,8546
3,4,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693
4,5,2025-01-01,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801
5,6,2025-01-01,320 YONGE,03:15,Wednesday,YONGE AND ELM,MFUS,0,0,None,0
6,7,2025-01-01,307 BATHURST,03:20,Wednesday,BATHURST AND DUNDAS,MFUS,0,0,N,0
7,8,2025-01-01,320 YONGE,03:23,Wednesday,YONGE AND SHUTER,MFSAN,5,10,N,3468
8,9,2025-01-01,112 WEST MALL,03:39,Wednesday,RENFORTH STATION,SFO,0,0,None,3306
9,10,2025-01-01,39 FINCH EAST,04:38,Wednesday,FINCH AND MCCOWAN,EFO,16,32,W,3144


In [179]:
delay_dataset.shape


(69037, 11)

So I picked the 4th row, that has a delay at OVERLEA AND THORNCLIFF	for 100 Flemingdon Park. I want to see if I can get to the Neighborhood it is from.

# TTC Routes and Schedules Data



I downloaded the data at:
https://open.toronto.ca/dataset/ttc-routes-and-schedules/


In [180]:
import pandas as pd
import requests
import io
from zipfile import ZipFile

# Example 1: Loading a local ZIP containing a CSV
# df = pd.read_csv('your_file.zip')

# Example 2: Loading from a URL (like the TTC Routes and Schedules data metadata found earlier)
zip_url = 'https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/7795b45e-e65a-4465-81fc-c36b9dfff169/resource/cfb6b2b8-6191-41e3-bda1-b175c51148cb/download/opendata_ttc_schedules.zip'

# Note: This specific TTC zip contains multiple text files (GTFS format).
# To load a specific file from inside the zip:
r = requests.get(zip_url)
with ZipFile(io.BytesIO(r.content)) as z:
    # List files in the zip
    print("Files in ZIP:", z.namelist())
    # Load 'routes.txt' which is a CSV-like format
    with z.open('routes.txt') as f:
        df_routes = pd.read_csv(f)
    with z.open('stops.txt') as f:
        df_route_stops = pd.read_csv(f)
    with z.open('stop_times.txt') as f:
        df_stop_times = pd.read_csv(f)
    with z.open('trips.txt') as f:
        df_trips = pd.read_csv(f)

display(df_routes.head())

Files in ZIP: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_times.txt', 'trips.txt']


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1,1,1,Line 1 (Yonge-University),NaN,1,NaN,D5C82B,000000
1,10,1,10,Van Horne,NaN,3,NaN,ED1C24,FFFFFF
2,100,1,100,Flemingdon Park,NaN,3,NaN,ED1C24,FFFFFF
3,101,1,101,Downsview Park,NaN,3,NaN,ED1C24,FFFFFF
4,102,1,102,Markham Rd,NaN,3,NaN,ED1C24,FFFFFF


So now I have the TTC routes and schedules, i can see if I can map the delay at OVERLEA and THORNCLIFFE to any route stops.

In [181]:
#df_route_stops.head()

In [182]:
# find an entry in df_route_stops with stop_name column
# having 'Overlea' and 'Thorncliff'
df_route_stops[df_route_stops['stop_name'].str.contains('Overlea') &
               df_route_stops['stop_name'].str.contains('Thorncliff')]

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
215,8011,8011,Overlea Blvd at Thorncliffe Park Dr (East),NaN,43.707250,-79.344028,NaN,NaN,NaN,NaN,NaN,1
486,8012,8012,Overlea Blvd at Thorncliffe Park Dr (West),NaN,43.705197,-79.349856,NaN,NaN,NaN,NaN,NaN,1
3663,4520,4520,Overlea Blvd at Thorncliffe Park Dr (West),NaN,43.704776,-79.349853,NaN,NaN,NaN,NaN,NaN,1
4108,8361,8361,Thorncliffe Park Dr East at Overlea Blvd South...,NaN,43.707006,-79.343507,NaN,NaN,NaN,NaN,NaN,1
6296,4518,4518,Overlea Blvd at Thorncliffe Park Dr (East) Wes...,NaN,43.707457,-79.344198,NaN,NaN,NaN,NaN,NaN,1
6990,8352,8352,Thorncliffe Park Dr (East) at Overlea Blvd,NaN,43.706926,-79.343188,NaN,NaN,NaN,NaN,NaN,1


In [183]:
df_route_stops[df_route_stops['stop_name'].str.contains('Warden Sta') ]

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
8150,13732,13732,Warden Station - Eastbound Platform,NaN,43.712149,-79.278785,NaN,NaN,NaN,NaN,NaN,1
8153,13731,13731,Warden Station - Westbound Platform,NaN,43.710949,-79.279085,NaN,NaN,NaN,NaN,NaN,1
9124,16636,16636,Warden Station,NaN,43.709817,-79.280007,NaN,NaN,NaN,NaN,NaN,1
9125,16639,16639,Warden Station,NaN,43.710364,-79.279471,NaN,NaN,NaN,NaN,NaN,1
9126,16640,16640,Warden Station,NaN,43.710463,-79.280181,NaN,NaN,NaN,NaN,NaN,1
9127,16642,16642,Warden Station,NaN,43.710938,-79.280430,NaN,NaN,NaN,NaN,NaN,1
9128,16643,16643,Warden Station,NaN,43.711198,-79.280529,NaN,NaN,NaN,NaN,NaN,1
9138,16638,16638,Warden Station,NaN,43.710158,-79.279509,NaN,NaN,NaN,NaN,NaN,1


The `delay_with_neighborhood` DataFrame now contains the original bus delay data along with `latitude`, `longitude`, `AREA_NAME`, and `AREA_SHORT_CODE` for each delay entry that could be successfully mapped to a geographic location and then to a Toronto neighborhood.

Note that some entries might have `NaN` for `AREA_NAME` and `AREA_SHORT_CODE` if their coordinates didn't fall within any of the defined neighborhood boundaries, or if no coordinates could be found for the original station name.

So as you can see, we found multiple bus stops at Overlead and Thorncliffe. So we can probably do more data cleaning to get the exact stop it was at, or we can take the mean longitude and latitude and use that to map it to a neighborhood.

# Toronto Neighborhoods

https://open.toronto.ca/dataset/neighbourhoods/

I downloaded this 158-neighbourhoods proile

https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/fc443770-ef0a-4025-9c2c-2cb558bfab00/resource/0719053b-28b7-48ea-b863-068823a93aaa/download/Neighbourhoods%20-%204326.geojson

I just used the longitude and latitude of stop_id 8011 (the first entry above) to see if it works.

In [184]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Load the Toronto 158-Neighborhood boundaries
gdf_neighborhoods = gpd.read_file(pathToFile + "Neighbourhoods - 4326.geojson")

# 2. Define your point (Longitude first for Shapely!)
lon, lat = -79.344028, 43.707250
my_point = Point(lon, lat)

# 3. Create a GeoDataFrame for your point(s)
gdf_point = gpd.GeoDataFrame([{'geometry': my_point}], crs="EPSG:4326")

# 4. Perform the Spatial Join
result = gpd.sjoin(gdf_point, gdf_neighborhoods, how="left", predicate="within")

# 5. Extract the Neighborhood Name and Number
neighborhood_name = result['AREA_NAME'].iloc[0]
# Usually 'AREA_SHORT_CODE' contains the ID/Number (e.g., 43)
neighborhood_number = result['AREA_SHORT_CODE'].iloc[0]

print(f"The coordinate maps to: {neighborhood_name} (Neighborhood #{neighborhood_number})")

# Display first few columns of the result to see all available attributes
display(result.head())

The coordinate maps to: Thorncliffe Park (Neighborhood #055)


,geometry,index_right,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID
0,POINT (-79.34403 43.70725),129,130,2502237,26022752,None,055,055,Thorncliffe Park,Thorncliffe Park (55),Neighbourhood Improvement Area,NIA,17826801.0


# Toronto Neighborhood Profile Data

Now I will load the Neighbourhood profile data, from the census 2021.

You must get the 158 neighbourhood model after 2021, else it is not current and won't match our transit data from 2025.

https://open.toronto.ca/dataset/neighbourhood-profiles/

In [185]:
df_hoods = pd.read_excel(pathToFile+"neighbourhood-profiles-2021-158-model.xlsx", skiprows= 0, header = 0, index_col = 0, sheet_name="hd2021_census_profile")

df_hoods.head(20)

,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling,Elms-Old Rexdale,Kingsview Village-The Westway,Willowridge-Martingrove-Richview,Humber Heights-Westmount,Edenbridge-Humber Valley,Princess-Rosethorn,Eringate-Centennial-West Deane,Markland Wood,Etobicoke West Mall,Kingsway South,Stonegate-Queensway,New Toronto,Long Branch,Alderwood,Humber Summit,Humbermede,Pelmo Park-Humberlea,Black Creek,Glenfield-Jane Heights,York University Heights,Rustic,Maple Leaf,Brookhaven-Amesbury,Yorkdale-Glen Park,Englemount-Lawrence,Clanton Park,Bathurst Manor,Westminster-Branson,Newtonbrook West,Willowdale West,Lansing-Westgate,Bedford Park-Nortown,St.Andrew-Windfields,Bridle Path-Sunnybrook-York Mills,Banbury-Don Mills,Victoria Village,Flemingdon Park,Pleasant View,Don Valley Village,Hillcrest Village,Bayview Woods-Steeles,Newtonbrook East,Bayview Village,Henry Farm,O`Connor Parkview,Thorncliffe Park,Leaside-Bennington,Broadview North,Old East York,Danforth-East York,Woodbine-Lumsden,Taylor Massey,East End Danforth,The Beaches,Woodbine Corridor,Greenwood-Coxwell,Danforth,Playter Estates-Danforth,North Riverdale,Blake-Jones,South Riverdale,Cabbagetown-South St. James Town,Regent Park,Moss Park,North St. James Town,Kensington-Chinatown,University,Palmerston-Little Italy,Trinity-Bellwoods,Dufferin Grove,Little Portugal,South Parkdale,Roncesvalles,High Park-Swansea,High Park North,Runnymede-Bloor West Village,Junction Area,Weston-Pelham Park,Corso Italia-Davenport,Wychwood,Annex,Casa Loma,Yonge-St. Clair,Rosedale-Moore Park,Mount Pleasant East,Yonge-Eglinton,Forest Hill South,Forest Hill North,Lawrence Park South,Lawrence Park North,Humewood-Cedarvale,Oakwood Village,Briar Hill-Belgravia,Caledonia-Fairbank,Keelesdale-Eglinton West,Rockcliffe-Smythe,Beechborough-Greenbrook,Weston,Lambton Baby Point,Mount Dennis,Steeles,Tam O'Shanter-Sullivan,Wexford/Maryvale,Clairlea-Birchmount,Oakridge,Birchcliffe-Cliffside,Cliffcrest,Kennedy Park,Ionview,Dorset Park,Agincourt South-Malvern West,Agincourt North,Milliken,Centennial Scarborough,Highland Creek,Morningside,West Hill,Eglinton East,Scarborough Village,Guildwood,Golfdale-Cedarbrae-Woburn,Woburn North,West Rouge,Morningside Heights,Malvern West,Malvern East,L'Amoreaux West,East L'Amoreaux,Parkwoods-O'Connor Hills,Fenside-Parkwoods,Yonge-Doris,East Willowdale,Avondale,Oakdale-Beverley Heights,Downsview,Bendale-Glen Andrew,Bendale South,Islington,Etobicoke City Centre,Mimico-Queensway,Humber Bay Shores,West Queen West,Fort York-Liberty Village,Wellington Place,Harbourfront-CityPlace,St Lawrence-East Bayfront-The Islands,Church-Wellesley,Downtown Yonge East,Bay-Cloverhill,Yonge-Bay Corridor,Junction-Wallace Emerson,Dovercourt Village,North Toronto,South Eglinton-Davisville
Neighbourhood Name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Neighbourhood Number,1,2,3,4,5,6,7,8,9,10,11,12,13,15,16,18,19,20,21,22,23,24,25,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,46,47,48,49,50,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,78,79,80,81,83,84,85,86,87,88,89,90,91,92,94,95,96,97,98,99,100,101,102,103,105,106,107,108,109,110,111,112,113,114,115,116,118,119,120,121,122,123,124,125,126,128,129,130,133,134,135,136,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174
TSNS 2020 Designation,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or 

In [186]:
# display hardcoded for now
df_hoods.iloc[:, 49]


,Thorncliffe Park
Neighbourhood Name,
Neighbourhood Number,55
TSNS 2020 Designation,Neighbourhood Improvement Area
Total - Age groups of the population - 25% sample data,20400
0 to 14 years,4975
0 to 4 years,1585
...,...
"Total - Eligibility and instruction in the minority official language, for the population in private households born between 2003 and 2015 (inclusive) - 25% sample data",4120
Children eligible for instruction in the minority official language,190
Eligible children who have been instructed in the minority official language at the primary or secondary level in Canada,120


There is too much data in the Census 2021 Data, so we will just select some rows we might be interested in, then transpose it before merging this data with the one with geometric data.

In [187]:
# too much extraneous data in the neighbourhood profile
# we won't use everything, so just keep the rows that we're
# interested in before we transpose it and add on to our
# geojson data

#drop rows that we don't care about
rows_to_keep = [0, 1, 2] # hood headings
rows_to_keep.extend([245, 246]) # income of household
rows_to_keep.extend([298,299,300]) # owner/renter
rows_to_keep.extend([1479, 1480, 1483]) # cdn citizen vs not
rows_to_keep.extend([1484, 1485, 1486]) # immigrants vs not
rows_to_keep.extend(list(range(2564,2595))) # commuting characteristics

df_hoods_subset = df_hoods.iloc[rows_to_keep]
df_hoods_subset.head(100)

,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling,Elms-Old Rexdale,Kingsview Village-The Westway,Willowridge-Martingrove-Richview,Humber Heights-Westmount,Edenbridge-Humber Valley,Princess-Rosethorn,Eringate-Centennial-West Deane,Markland Wood,Etobicoke West Mall,Kingsway South,Stonegate-Queensway,New Toronto,Long Branch,Alderwood,Humber Summit,Humbermede,Pelmo Park-Humberlea,Black Creek,Glenfield-Jane Heights,York University Heights,Rustic,Maple Leaf,Brookhaven-Amesbury,Yorkdale-Glen Park,Englemount-Lawrence,Clanton Park,Bathurst Manor,Westminster-Branson,Newtonbrook West,Willowdale West,Lansing-Westgate,Bedford Park-Nortown,St.Andrew-Windfields,Bridle Path-Sunnybrook-York Mills,Banbury-Don Mills,Victoria Village,Flemingdon Park,Pleasant View,Don Valley Village,Hillcrest Village,Bayview Woods-Steeles,Newtonbrook East,Bayview Village,Henry Farm,O`Connor Parkview,Thorncliffe Park,Leaside-Bennington,Broadview North,Old East York,Danforth-East York,Woodbine-Lumsden,Taylor Massey,East End Danforth,The Beaches,Woodbine Corridor,Greenwood-Coxwell,Danforth,Playter Estates-Danforth,North Riverdale,Blake-Jones,South Riverdale,Cabbagetown-South St. James Town,Regent Park,Moss Park,North St. James Town,Kensington-Chinatown,University,Palmerston-Little Italy,Trinity-Bellwoods,Dufferin Grove,Little Portugal,South Parkdale,Roncesvalles,High Park-Swansea,High Park North,Runnymede-Bloor West Village,Junction Area,Weston-Pelham Park,Corso Italia-Davenport,Wychwood,Annex,Casa Loma,Yonge-St. Clair,Rosedale-Moore Park,Mount Pleasant East,Yonge-Eglinton,Forest Hill South,Forest Hill North,Lawrence Park South,Lawrence Park North,Humewood-Cedarvale,Oakwood Village,Briar Hill-Belgravia,Caledonia-Fairbank,Keelesdale-Eglinton West,Rockcliffe-Smythe,Beechborough-Greenbrook,Weston,Lambton Baby Point,Mount Dennis,Steeles,Tam O'Shanter-Sullivan,Wexford/Maryvale,Clairlea-Birchmount,Oakridge,Birchcliffe-Cliffside,Cliffcrest,Kennedy Park,Ionview,Dorset Park,Agincourt South-Malvern West,Agincourt North,Milliken,Centennial Scarborough,Highland Creek,Morningside,West Hill,Eglinton East,Scarborough Village,Guildwood,Golfdale-Cedarbrae-Woburn,Woburn North,West Rouge,Morningside Heights,Malvern West,Malvern East,L'Amoreaux West,East L'Amoreaux,Parkwoods-O'Connor Hills,Fenside-Parkwoods,Yonge-Doris,East Willowdale,Avondale,Oakdale-Beverley Heights,Downsview,Bendale-Glen Andrew,Bendale South,Islington,Etobicoke City Centre,Mimico-Queensway,Humber Bay Shores,West Queen West,Fort York-Liberty Village,Wellington Place,Harbourfront-CityPlace,St Lawrence-East Bayfront-The Islands,Church-Wellesley,Downtown Yonge East,Bay-Cloverhill,Yonge-Bay Corridor,Junction-Wallace Emerson,Dovercourt Village,North Toronto,South Eglinton-Davisville
Neighbourhood Name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Neighbourhood Number,1,2,3,4,5,6,7,8,9,10,11,12,13,15,16,18,19,20,21,22,23,24,25,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,46,47,48,49,50,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,78,79,80,81,83,84,85,86,87,88,89,90,91,92,94,95,96,97,98,99,100,101,102,103,105,106,107,108,109,110,111,112,113,114,115,116,118,119,120,121,122,123,124,125,126,128,129,130,133,134,135,136,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174
TSNS 2020 Designation,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or 

In [188]:
df_hoods_subset_tr = df_hoods_subset.transpose()
df_hoods_subset_tr.shape


(158, 45)

So we only kept around 45 features from the Census.

In [189]:
df_hoods_subset_tr.head()

Neighbourhood Name,Neighbourhood Number,TSNS 2020 Designation,Total - Age groups of the population - 25% sample data,Median after-tax income of household in 2020 ($),Average after-tax income of household in 2020 ($),Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Citizenship for the population in private households - 25% sample data,Canadian citizens,Not Canadian citizens,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,Immigrants,Total - Place of work status for the employed labour force aged 15 years and over - 25% sample data,Worked at home,Worked outside Canada,No fixed workplace address,Usual place of work,Total - Commuting destination for the employed labour force aged 15 years and over with a usual place of work - 25% sample data,Commute within census subdivision (CSD) of residence,Commute to a different census subdivision (CSD) within census division (CD) of residence,Commute to a different census subdivision (CSD) and census division (CD) within province or territory of residence,Commute to a different province or territory,Total - Main mode of commuting for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,"Car, truck or van","Car, truck or van - as a driver","Car, truck or van - as a passenger",Public transit,Walked,Bicycle,Other method,Total - Commuting duration for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Less than 15 minutes,15 to 29 minutes,30 to 44 minutes,45 to 59 minutes,60 minutes and over,Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Between 5 a.m. and 5:59 a.m.,Between 6 a.m. and 6:59 a.m.,Between 7 a.m. and 7:59 a.m.,Between 8 a.m. and 8:59 a.m.,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.
West Humber-Clairville,1,Not an NIA or Emerging Neighbourhood,33300,83000,92300,10700,6995,3705,33300,26285,7010,33300,11805,18805,15780,2815,40,2630,10290,10290,6185,0,4065,40,12920,9260,8190,1075,3110,255,80,215,12920,2305,4390,3400,1125,1695,12920,980,2445,2630,2270,1665,2935
Mount Olive-Silverstone-Jamestown,2,Neighbourhood Improvement Area,31345,70500,78200,9740,4485,5260,31345,23340,8005,31345,9620,19810,11830,1655,45,1910,8215,8215,4740,0,3465,10,10125,7400,6395,1005,2235,255,10,220,10125,1655,3275,2845,1000,1345,10125,785,1830,1855,1540,1145,2965
Thistletown-Beaumond Heights,3,Neighbourhood Improvement Area,9850,78000,88800,3245,2025,1220,9850,8045,1805,9850,4055,5210,4165,775,25,845,2515,2515,1750,0,760,0,3365,2530,2255,280,660,45,20,110,3365,550,1135,855,275,550,3365,245,680,745,660,395,635
Rexdale-Kipling,4,Not an NIA or Emerging Neighbourhood,10375,71000,79200,3945,1990,1955,10375,8855,1520,10375,5080,4820,4560,870,10,810,2875,2880,1965,0,905,10,3680,2660,2360,300,830,105,10,80,3680,590,1295,915,330,560,3680,340,695,775,665,425,775
Elms-Old Rexdale,5,Neighbourhood Improvement Area,9355,74500,83600,3195,1760,1430,9355,8200,1155,9355,4515,4600,3600,630,10,625,2345,2345,1640,0,695,0,2965,2170,1920,245,615,85,0,85,2965,380,1185,780,260,360,2965,250,585,590,510,355,675


In [190]:
gdf_neighborhoods.head()


,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry
0,1,2502366,26022881,None,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NA,17824737.0,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ..."
1,2,2502365,26022880,None,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NA,17824753.0,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ..."
2,3,2502364,26022879,None,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NA,17824769.0,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ..."
3,4,2502363,26022878,None,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NA,17824785.0,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4..."
4,5,2502362,26022877,None,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NA,17824801.0,"MULTIPOLYGON (((-79.38404 43.64497, -79.38502 ..."


In [191]:
# combine the selected info from the census in the one with
# geographic data

# first need to convert AREA_SHORT_CODE to numbers
gdf_neighborhoods_cleaned = gdf_neighborhoods.copy()
pd.to_numeric(gdf_neighborhoods_cleaned['AREA_SHORT_CODE'], errors='coerce')
gdf_neighborhoods_cleaned.dropna(subset=['AREA_SHORT_CODE'], inplace=True)
gdf_neighborhoods_cleaned['AREA_SHORT_CODE'] = gdf_neighborhoods_cleaned['AREA_SHORT_CODE'].astype(int)

# then convert hoods neighbourhood numbers to numbers as well
df_hoods_subset_tr_cleaned = df_hoods_subset_tr.copy()
pd.to_numeric(df_hoods_subset_tr_cleaned['Neighbourhood Number'], errors='coerce')
df_hoods_subset_tr_cleaned.dropna(subset=['Neighbourhood Number'], inplace=True)
df_hoods_subset_tr_cleaned['Neighbourhood Number'] = df_hoods_subset_tr_cleaned['Neighbourhood Number'].astype(int)

gdf_neighborhoods_full = gdf_neighborhoods_cleaned.merge(df_hoods_subset_tr_cleaned, left_on='AREA_SHORT_CODE', right_on='Neighbourhood Number')

In [192]:
gdf_neighborhoods_full.shape

(158, 57)

In [193]:
gdf_neighborhoods_full.head()

,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry,Neighbourhood Number,TSNS 2020 Designation,Total - Age groups of the population - 25% sample data,Median after-tax income of household in 2020 ($),Average after-tax income of household in 2020 ($),Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Citizenship for the population in private households - 25% sample data,Canadian citizens,Not Canadian citizens,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,Immigrants,Total - Place of work status for the employed labour force aged 15 years and over - 25% sample data,Worked at home,Worked outside Canada,No fixed workplace address,Usual place of work,Total - Commuting destination for the employed labour force aged 15 years and over with a usual place of work - 25% sample data,Commute within census subdivision (CSD) of residence,Commute to a different census subdivision (CSD) within census division (CD) of residence,Commute to a different census subdivision (CSD) and census division (CD) within province or territory of residence,Commute to a different province or territory,Total - Main mode of commuting for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,"Car, truck or van","Car, truck or van - as a driver","Car, truck or van - as a passenger",Public transit,Walked,Bicycle,Other method,Total - Commuting duration for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Less than 15 minutes,15 to 29 minutes,30 to 44 minutes,45 to 59 minutes,60 minutes and over,Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Between 5 a.m. and 5:59 a.m.,Between 6 a.m. and 6:59 a.m.,Between 7 a.m. and 7:59 a.m.,Between 8 a.m. and 8:59 a.m.,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.
0,1,2502366,26022881,None,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NA,17824737.0,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ...",174,Not an NIA or Emerging Neighbourhood,22735,68500,82100,13185,4010,9170,22735,16525,6210,22735,11005,9360,13475,7330,95,855,5195,5200,4560,0,610,30,6055,2410,2155,255,2655,665,110,215,6055,930,1905,2005,665,550,6055,175,600,1360,1800,1390,730
1,2,2502365,26022880,None,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NA,17824753.0,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ...",173,Not an NIA or Emerging Neighbourhood,15885,61200,70900,9430,1750,7680,15885,10945,4945,15885,6625,7135,9460,4890,35,660,3880,3880,3315,0,555,10,4540,1760,1565,195,1950,545,150,145,4540,640,1220,1670,550,465,4540,140,570,1060,1255,970,550
2,3,2502364,26022879,None,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NA,17824769.0,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ...",172,Not an NIA or Emerging Neighbourhood,12380,77000,92300,5310,2900,2410,12380,11045,1340,12380,7450,4445,6430,2910,20,720,2775,2775,2495,0,270,0,3495,1645,1425,215,965,390,395,105,3495,600,1115,1115,385,285,3495,185,460,690,825,775,570
3,4,2502363,26022878,None,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NA,17824785.0,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4...",171,Not an NIA or Emerging Neighbourhood,23180,75000,86700,10185,5105,5080,23180,19960,3225,23180,13320,9020,12070,5135,20,1660,5260,5260,4620,0,635,0,6925,3430,2945,485,2275,610,350,270,6925,1015,2135,2225,845,715,6925,475,1195,1335,1635,1185,1105
4,5,2502362,26022877,None,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NA,17824801.0,"MU

# Data Generation

So now we can perform a spatial join with the TTC bus delay, and build our data.

In [194]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

# --- Step 1: Create a mapping of Station names to coordinates ---

# Clean df_route_stops stop_name for better matching
df_route_stops['cleaned_stop_name'] = df_route_stops['stop_name'].str.upper().str.replace(r'[^A-Z0-9\s]', '', regex=True).str.strip()

# Initialize dictionary to store mappings from delay_dataset station names to (lat, lon)
station_to_coords = {}

# Get unique station names from delay_dataset to process them efficiently
unique_delay_stations = delay_dataset['Station'].unique()

# Iterate through each unique delay station name to find its coordinates
for delay_station in unique_delay_stations:
    # Clean the delay station name
    # Attempt to generalize keywords from delay station names
    cleaned_delay_station = delay_station.upper()
    # Replace ' AND ' with space to treat as separate keywords
    cleaned_delay_station = cleaned_delay_station.replace(' AND ', ' ')
    # Remove common suffixes/words that might not appear in stop_name consistently
    cleaned_delay_station = cleaned_delay_station.replace(' STATION', '').replace(' INTERSECTION', '').replace(' AT ', ' ').replace(' BLVD', '').replace(' DR', '').strip()
    # Split into keywords
    keywords = [kw.strip() for kw in cleaned_delay_station.split() if kw.strip()]

    if not keywords: # Handle cases where cleaning results in an empty string
        station_to_coords[delay_station] = (np.nan, np.nan)
        continue

    # Filter df_route_stops where all keywords are present in the cleaned_stop_name
    # This allows for partial matches and flexible ordering of keywords
    mask = df_route_stops['cleaned_stop_name'].apply(lambda x: all(kw in x for kw in keywords))
    matching_stops = df_route_stops[mask]

    if not matching_stops.empty:
        # Calculate mean lat/lon for all matching stops
        mean_lat = matching_stops['stop_lat'].mean()
        mean_lon = matching_stops['stop_lon'].mean()
        station_to_coords[delay_station] = (mean_lat, mean_lon)
    else:
        # If no match found, store NaN
        station_to_coords[delay_station] = (np.nan, np.nan)

print(f"Created coordinate mapping for {len(station_to_coords)} unique delay stations.")


Created coordinate mapping for 12254 unique delay stations.


In [195]:
# --- Step 2: Add latitude and longitude columns to delay_dataset ---
# Using .copy() to avoid SettingWithCopyWarning
delay_dataset_with_coords = delay_dataset.copy()

delay_dataset_with_coords['latitude'] = delay_dataset_with_coords['Station'].map(lambda x: station_to_coords.get(x, (np.nan, np.nan))[0])
delay_dataset_with_coords['longitude'] = delay_dataset_with_coords['Station'].map(lambda x: station_to_coords.get(x, (np.nan, np.nan))[1])

# Filter out rows where latitude or longitude could not be determined
# These are delay entries for which no matching stop could be found in df_route_stops
delay_dataset_with_coords = delay_dataset_with_coords.dropna(subset=['latitude', 'longitude'])

print(f"Original delay_dataset had {delay_dataset.shape[0]} rows.")
print(f"After mapping coordinates, {delay_dataset_with_coords.shape[0]} rows have valid coordinates.")

# --- Step 3: Create a GeoDataFrame from delay_dataset_with_coords ---
gdf_delay_points = gpd.GeoDataFrame(
    delay_dataset_with_coords,
    geometry=gpd.points_from_xy(delay_dataset_with_coords.longitude, delay_dataset_with_coords.latitude),
    crs="EPSG:4326" # Ensure CRS matches the neighborhoods GeoDataFrame
)

print(f"Created GeoDataFrame with {gdf_delay_points.shape[0]} delay points.")


Original delay_dataset had 69037 rows.
After mapping coordinates, 57607 rows have valid coordinates.
Created GeoDataFrame with 57607 delay points.


At this point, I'll just disregard the 12000 rows that I can't map.  I'll work on that later.

In [196]:
gdf_neighborhoods_full.head()

,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry,Neighbourhood Number,TSNS 2020 Designation,Total - Age groups of the population - 25% sample data,Median after-tax income of household in 2020 ($),Average after-tax income of household in 2020 ($),Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Citizenship for the population in private households - 25% sample data,Canadian citizens,Not Canadian citizens,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,Immigrants,Total - Place of work status for the employed labour force aged 15 years and over - 25% sample data,Worked at home,Worked outside Canada,No fixed workplace address,Usual place of work,Total - Commuting destination for the employed labour force aged 15 years and over with a usual place of work - 25% sample data,Commute within census subdivision (CSD) of residence,Commute to a different census subdivision (CSD) within census division (CD) of residence,Commute to a different census subdivision (CSD) and census division (CD) within province or territory of residence,Commute to a different province or territory,Total - Main mode of commuting for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,"Car, truck or van","Car, truck or van - as a driver","Car, truck or van - as a passenger",Public transit,Walked,Bicycle,Other method,Total - Commuting duration for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Less than 15 minutes,15 to 29 minutes,30 to 44 minutes,45 to 59 minutes,60 minutes and over,Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Between 5 a.m. and 5:59 a.m.,Between 6 a.m. and 6:59 a.m.,Between 7 a.m. and 7:59 a.m.,Between 8 a.m. and 8:59 a.m.,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.
0,1,2502366,26022881,None,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NA,17824737.0,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ...",174,Not an NIA or Emerging Neighbourhood,22735,68500,82100,13185,4010,9170,22735,16525,6210,22735,11005,9360,13475,7330,95,855,5195,5200,4560,0,610,30,6055,2410,2155,255,2655,665,110,215,6055,930,1905,2005,665,550,6055,175,600,1360,1800,1390,730
1,2,2502365,26022880,None,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NA,17824753.0,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ...",173,Not an NIA or Emerging Neighbourhood,15885,61200,70900,9430,1750,7680,15885,10945,4945,15885,6625,7135,9460,4890,35,660,3880,3880,3315,0,555,10,4540,1760,1565,195,1950,545,150,145,4540,640,1220,1670,550,465,4540,140,570,1060,1255,970,550
2,3,2502364,26022879,None,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NA,17824769.0,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ...",172,Not an NIA or Emerging Neighbourhood,12380,77000,92300,5310,2900,2410,12380,11045,1340,12380,7450,4445,6430,2910,20,720,2775,2775,2495,0,270,0,3495,1645,1425,215,965,390,395,105,3495,600,1115,1115,385,285,3495,185,460,690,825,775,570
3,4,2502363,26022878,None,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NA,17824785.0,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4...",171,Not an NIA or Emerging Neighbourhood,23180,75000,86700,10185,5105,5080,23180,19960,3225,23180,13320,9020,12070,5135,20,1660,5260,5260,4620,0,635,0,6925,3430,2945,485,2275,610,350,270,6925,1015,2135,2225,845,715,6925,475,1195,1335,1635,1185,1105
4,5,2502362,26022877,None,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NA,17824801.0,"MU

In [197]:
# --- Step 4: Perform the Spatial Join with neighborhoods ---
# Perform a left spatial join to keep all delay entries that have coordinates,
# and append neighborhood information if a point falls within a neighborhood.
# If a point doesn't fall within any neighborhood, 'AREA_NAME' and 'AREA_SHORT_CODE' will be NaN.
delay_with_neighborhood = gpd.sjoin(gdf_delay_points, gdf_neighborhoods_full, how="left", predicate="within")

# Drop the 'index_right' column which is a byproduct of the sjoin operation and usually not needed
delay_with_neighborhood = delay_with_neighborhood.drop(columns=['index_right'])

# --- Step 5: Review results ---
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 20)

display(delay_with_neighborhood.head())

print(f"\nFinal shape of delay data joined with neighborhood information: {delay_with_neighborhood.shape}")
print(f"Number of delay entries successfully mapped to a specific neighborhood: {delay_with_neighborhood['AREA_NAME'].notna().sum()}")
print(f"Number of delay entries with coordinates but not falling into a defined neighborhood: {delay_with_neighborhood['AREA_NAME'].isna().sum()}")


,_id_left,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle,latitude,longitude,geometry,_id_right,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,Neighbourhood Number,TSNS 2020 Designation,Total - Age groups of the population - 25% sample data,Median after-tax income of household in 2020 ($),Average after-tax income of household in 2020 ($),Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Citizenship for the population in private households - 25% sample data,Canadian citizens,Not Canadian citizens,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,Immigrants,Total - Place of work status for the employed labour force aged 15 years and over - 25% sample data,Worked at home,Worked outside Canada,No fixed workplace address,Usual place of work,Total - Commuting destination for the employed labour force aged 15 years and over with a usual place of work - 25% sample data,Commute within census subdivision (CSD) of residence,Commute to a different census subdivision (CSD) within census division (CD) of residence,Commute to a different census subdivision (CSD) and census division (CD) within province or territory of residence,Commute to a different province or territory,Total - Main mode of commuting for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,"Car, truck or van","Car, truck or van - as a driver","Car, truck or van - as a passenger",Public transit,Walked,Bicycle,Other method,Total - Commuting duration for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Less than 15 minutes,15 to 29 minutes,30 to 44 minutes,45 to 59 minutes,60 minutes and over,Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Between 5 a.m. and 5:59 a.m.,Between 6 a.m. and 6:59 a.m.,Between 7 a.m. and 7:59 a.m.,Between 8 a.m. and 8:59 a.m.,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.
0,1,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442,43.769610,-79.304280,POINT (-79.30428 43.76961),70.0,2502297.0,26022812.0,None,119.0,119,Wexford/Maryvale,Wexford/Maryvale (119),Not an NIA or Emerging Neighbourhood,NA,17825841.0,119.0,Not an NIA or Emerging Neighbourhood,28345,74000,85800,10260,6050,4220,28340,23620,4720,28340,12365,14400,12445,3120,25,1805,7490,7490,6240,0,1240,10,9300,6300,5485,815,2290,405,35,265,9300,1815,2840,2395,975,1270,9300,650,1285,1905,1845,1665,1950
1,2,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,None,0,43.676448,-79.551377,POINT (-79.55138 43.67645),145.0,2502222.0,26022737.0,None,10.0,010,Princess-Rosethorn,Princess-Rosethorn (10),Not an NIA or Emerging Neighbourhood,NA,17827041.0,10.0,Not an NIA or Emerging Neighbourhood,11170,133000,164200,3900,3310,590,11170,10460,710,11170,7615,3400,5215,2260,15,490,2450,2450,1770,0,675,0,2940,2435,2230,205,290,95,15,110,2940,640,1085,705,270,240,2940,105,360,730,865,625,260
2,3,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,None,8546,43.674153,-79.355380,POINT (-79.35538 43.67415),88.0,2502279.0,26022794.0,None,68.0,068,North Riverdale,North Riverdale (68),Not an NIA or Emerging Neighbourhood,NA,17826129.0,68.0,Not an NIA or Emerging Neighbourhood,11290,93000,127900,4905,2855,2045,11290,10680,610,11290,8395,2720,5760,3145,25,395,2205,2205,2010,0,175,20,2600,1155,1020,130,615,465,270,90,2600,460,1010,725,215,190,2600,40,280,690,770,600,225
3,4,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693,43.706435,-79.345772,POINT (-79.34577 43.70644),130.0,2502237.0,26022752.0,None,55.0,055,Thorncliffe Park,Thorncliffe Park (55),Neighbourhood Improvement A


Final shape of delay data joined with neighborhood information: (57607, 70)
Number of delay entries successfully mapped to a specific neighborhood: 56206
Number of delay entries with coordinates but not falling into a defined neighborhood: 1401


In [198]:
delay_with_neighborhood.describe()


,_id_left,Date,Min Delay,Min Gap,Vehicle,latitude,longitude,_id_right,AREA_ID,AREA_ATTR_ID,AREA_SHORT_CODE,OBJECTID,Neighbourhood Number
count,57607.000000,57607,57607.000000,57607.000000,57607.000000,57607.000000,57607.000000,56206.000000,5.620600e+04,5.620600e+04,56206.000000,5.620600e+04,56206.000000
mean,34433.233791,2025-07-24 18:56:47.210929152,19.785321,32.107851,5991.787456,43.728998,-79.389167,73.271537,2.502294e+06,2.602281e+07,92.266004,1.782589e+07,92.266004
min,1.000000,2025-01-01 00:00:00,0.000000,0.000000,0.000000,43.591961,-79.649837,1.000000,2.502209e+06,2.602272e+07,1.000000,1.782474e+07,1.000000
25%,17321.500000,2025-04-20 00:00:00,8.000000,16.000000,3334.000000,43.688775,-79.475772,30.000000,2.502248e+06,2.602276e+07,39.000000,1.782520e+07,39.000000
50%,34441.000000,2025-07-26 00:00:00,12.000000,22.000000,7121.000000,43.731833,-79.392350,70.000000,2.502297e+06,2.602281e+07,107.000000,1.782584e+07,107.000000
75%,51564.500000,2025-10-31 00:00:00,20.000000,40.000000,8615.500000,43.769269,-79.297344,119.000000,2.502337e+06,2.602285e+07,142.000000,1.782662e+07,142.000000
max,69036.000000,2026-01-31 00:00:00,999.000000,997.000000,94633.000000,43.909707,-79.123111,158.000000,2.502366e+06,2.602288e+07,174.000000,1.782725e+07,174.000000
std,19867.615104,NaN,46.706590,48.791100,3703.852752,0.051271,0.109436,48.896324,4.889632e+01,4.889632e+01,53.214699,7.823412e+02,53.214699


# Data Cleanup

There are a lot of unnecessary columns, so I'll drop them here.

Also, some columns need to be normalized, and we should do that here as well


In [199]:
cols_to_drop = ['_id_left', 'geometry', '_id_right','AREA_ID',
                'AREA_ATTR_ID','PARENT_AREA_ID', 'AREA_LONG_CODE',
                'OBJECTID', 'Neighbourhood Number',
                'TSNS 2020 Designation']

delay_with_neighborhood.drop(columns=cols_to_drop, inplace=True)

In [200]:
delay_with_neighborhood.shape


(57607, 60)

In [201]:
print(delay_with_neighborhood.dtypes)

Date                                                                                                                                                              datetime64[ns]
Line                                                                                                                                                                      object
Time                                                                                                                                                                      object
Day                                                                                                                                                                       object
Station                                                                                                                                                                   object
Code                                                                                                               

In [202]:
# convert the AREA_SHORT_CODE to int64
print(delay_with_neighborhood['AREA_SHORT_CODE'].isna().sum() )

# currently just drop the rows that we can't map to a neighborhood
delay_with_neighborhood.dropna(subset=['AREA_SHORT_CODE'], inplace=True)

1401


In [203]:
delay_with_neighborhood.shape

(56206, 60)

In [204]:
# convert AREA_SHORT_CODE from float to int
delay_with_neighborhood['AREA_SHORT_CODE'].astype('int64')


,AREA_SHORT_CODE
0,119
1,10
2,68
3,55
4,42
...,...
69030,111
69031,4
69032,112
69033,126


# Writing Data out to Excel File


In [205]:
# write out delay_with_neighborhood to a excel file so others can use it
delay_with_neighborhood.to_excel(pathToGroup + "/delay_with_neighborhood.xlsx")

# writing out a json file gave me problems
#delay_with_neighborhood.to_file(pathToGroup + "/delay_neighborhoods.json", driver="GeoJSON")

In [206]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 30)

delay_with_neighborhood.head(10)

,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle,latitude,longitude,AREA_SHORT_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,Total - Age groups of the population - 25% sample data,Median after-tax income of household in 2020 ($),Average after-tax income of household in 2020 ($),Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Citizenship for the population in private households - 25% sample data,Canadian citizens,Not Canadian citizens,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,Immigrants,Total - Place of work status for the employed labour force aged 15 years and over - 25% sample data,Worked at home,Worked outside Canada,No fixed workplace address,Usual place of work,Total - Commuting destination for the employed labour force aged 15 years and over with a usual place of work - 25% sample data,Commute within census subdivision (CSD) of residence,Commute to a different census subdivision (CSD) within census division (CD) of residence,Commute to a different census subdivision (CSD) and census division (CD) within province or territory of residence,Commute to a different province or territory,Total - Main mode of commuting for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,"Car, truck or van","Car, truck or van - as a driver","Car, truck or van - as a passenger",Public transit,Walked,Bicycle,Other method,Total - Commuting duration for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Less than 15 minutes,15 to 29 minutes,30 to 44 minutes,45 to 59 minutes,60 minutes and over,Total - Time leaving for work for the employed labour force aged 15 years and over with a usual place of work or no fixed workplace address - 25% sample data,Between 5 a.m. and 5:59 a.m.,Between 6 a.m. and 6:59 a.m.,Between 7 a.m. and 7:59 a.m.,Between 8 a.m. and 8:59 a.m.,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.
0,2025-01-01,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442,43.769610,-79.304280,119.0,Wexford/Maryvale,Wexford/Maryvale (119),Not an NIA or Emerging Neighbourhood,NA,28345,74000,85800,10260,6050,4220,28340,23620,4720,28340,12365,14400,12445,3120,25,1805,7490,7490,6240,0,1240,10,9300,6300,5485,815,2290,405,35,265,9300,1815,2840,2395,975,1270,9300,650,1285,1905,1845,1665,1950
1,2025-01-01,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,None,0,43.676448,-79.551377,10.0,Princess-Rosethorn,Princess-Rosethorn (10),Not an NIA or Emerging Neighbourhood,NA,11170,133000,164200,3900,3310,590,11170,10460,710,11170,7615,3400,5215,2260,15,490,2450,2450,1770,0,675,0,2940,2435,2230,205,290,95,15,110,2940,640,1085,705,270,240,2940,105,360,730,865,625,260
2,2025-01-01,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,None,8546,43.674153,-79.355380,68.0,North Riverdale,North Riverdale (68),Not an NIA or Emerging Neighbourhood,NA,11290,93000,127900,4905,2855,2045,11290,10680,610,11290,8395,2720,5760,3145,25,395,2205,2205,2010,0,175,20,2600,1155,1020,130,615,465,270,90,2600,460,1010,725,215,190,2600,40,280,690,770,600,225
3,2025-01-01,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693,43.706435,-79.345772,55.0,Thorncliffe Park,Thorncliffe Park (55),Neighbourhood Improvement Area,NIA,20400,62400,68500,7065,990,6070,20400,15215,5185,20400,6420,12905,7140,2130,50,1330,3630,3630,3070,0,550,10,4960,2880,2590,285,1465,405,40,165,4960,925,1285,1380,715,655,4960,245,640,985,920,940,1230
4,2025-01-01,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801,43.720512,-79.338919,42.0,Banbury-Don Mills,Banbury-Don Mills (42),Not an NIA or Emerging Neighbourhood,NA,27155,78000,109700,12215,7295,4915,27150,23375,3780,27150,12640,13675,12325,5305,190,980,5850,5855,4940,0,895,15,6835,5015,4540,480,1145,335,65,270